# ChakraTransformer Colab Evaluation & Video Inference

This notebook evaluates the pre-trained ChakraTransformer (Combo 6) on the Kvasir-SEG test set and runs video inference using Google Drive.

**Instructions:**
1. Extract the `kaggle_upload.zip` file and upload the contents to a folder in your Google Drive (e.g., `MyDrive/chakramodel`).
2. Upload your testing videos (`.mp4` or `.avi`) into that same folder (or a subfolder).
3. Run the cells below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip install -q timm opencv-python torchvision

import sys
import os

# Set this path to where you extracted the zip in your Google Drive
BASE_DIR = '/content/drive/MyDrive/chakramodel_necessary_zip'

if not os.path.exists(BASE_DIR):
    print(f"WARNING: BASE_DIR '{BASE_DIR}' not found.")
    print("Please update BASE_DIR to match your Google Drive path.")
else:
    # Add src to sys.path so we can import the modules natively from Drive
    sys.path.append(f"{BASE_DIR}/src")
    print("Source code successfully linked from Google Drive.")


In [ ]:
import torch
from pathlib import Path
import cv2
import numpy as np
from torchvision import transforms
import time
from torch.utils.data import DataLoader

from chakra_transformer.transformer_segmenter import ChakraTransformerSegmenter
from train_pranet import KvasirSEGDataset
from evaluate_all import evaluate_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


In [ ]:
# Load Model
print("Loading ChakraTransformer...")
model = ChakraTransformerSegmenter(backbone_name='vit_large_patch16_384', pretrained=False)

weight_path = f"{BASE_DIR}/chakra_transformer_best.pth"
if not os.path.exists(weight_path):
    weight_path = f"{BASE_DIR}/weights/chakra_transformer_best.pth"

model.load_state_dict(torch.load(weight_path, map_location=device))
model.to(device).eval()
print(f"Model weights loaded successfully from: {weight_path}")


## 1. Video Inference from Google Drive
Run the ChakraTransformer on all `.mp4` and `.avi` videos found in the video directory.

In [ ]:
# We will scan this directory for any videos to test
video_dir = Path('/content/drive/MyDrive/chakramodel_necessary_zip/video for testing')
output_dir = Path('/content/output_videos')
output_dir.mkdir(parents=True, exist_ok=True)

video_files = list(video_dir.glob('*.mp4')) + list(video_dir.glob('*.avi'))
print(f"Found {len(video_files)} videos for testing in {video_dir}.")

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((448, 448)),
    transforms.ToTensor(),
])

for vid_path in video_files:
    print(f"\nProcessing {vid_path.name}...")
    cap = cv2.VideoCapture(str(vid_path))
    
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    
    out_path = str(output_dir / f"annotated_{vid_path.name}")
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(out_path, fourcc, int(fps), (width, height))
    
    frame_count = 0
    start_time = time.time()
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        input_tensor = transform(rgb_frame).unsqueeze(0).to(device)
        
        with torch.no_grad():
            outputs = model(input_tensor)
            if isinstance(outputs, (tuple, list)):
                pred = outputs[0]
            else:
                pred = outputs
            pred = torch.sigmoid(pred).squeeze().cpu().numpy()
            
        pred_resized = cv2.resize(pred, (width, height))
        mask = (pred_resized > 0.5).astype(np.uint8) * 255
        
        colored_mask = np.zeros_like(frame)
        colored_mask[:, :, 1] = mask
        
        overlay = cv2.addWeighted(frame, 0.7, colored_mask, 0.3, 0)
        out.write(overlay)
        
        frame_count += 1
        if frame_count % 30 == 0:
            print(f"  Processed {frame_count} frames...")
            
    cap.release()
    out.release()
    
    if frame_count > 0:
        elapsed = time.time() - start_time
        print(f"Finished {vid_path.name} - {frame_count} frames in {elapsed:.2f}s ({frame_count/elapsed:.1f} FPS)")
        print(f"Saved to {out_path}. Copying to Drive...")
        
        # Copy the annotated video back to Google Drive
        drive_out_dir = video_dir / 'output_videos'
        drive_out_dir.mkdir(parents=True, exist_ok=True)
        !cp "{out_path}" "{drive_out_dir}/annotated_{vid_path.name}"
        print(f"Copied to {drive_out_dir}/annotated_{vid_path.name}")
    else:
        print(f"Failed to process any frames from {vid_path.name}.")
